In [1]:
# import the necessary libraries
import ollama
import pandas as pd
import os
import time
from tqdm import tqdm

## Setting up the file path for financial sarcasm data and the newly labelled data for experiment 1

In [2]:

finance_data = "/Users/mariyajose/Documents/Sarcasm/DeepSeek/rq2/sarcasm_test_data.csv"
exp1_labeled = "/Users/mariyajose/Documents/Sarcasm/DeepSeek/rq2/exp3_labeled.csv"



In [3]:
# read the finance data into pandas dataframe fin_data
fin_data = pd.read_csv(finance_data)
print(f"Total rows before removing nulls: {len(fin_data)}")

Total rows before removing nulls: 590


## Preschesks before starting the process
#### 1. Check if there any file with the same name.If it exists check how many rows are already labelled and labbel the remaining ones
#### 2. if the file does ot exists start labelling the data from the beginning 


In [4]:
if os.path.exists(exp1_labeled):
    print("Found an existing file with the same name, continuing labeling...")
    new_labeled_df = pd.read_csv(exp1_labeled)
    
    if 'exp1_label' not in new_labeled_df.columns:
        new_labeled_df['exp1_label'] = pd.NA
else:
    print("File not found with the given name, creating a new file and starting labeling from the beginning")
    new_labeled_df = fin_data.copy()
    new_labeled_df['exp1_label'] = pd.NA

# Fixed index finder to handle pd.NA safely
def index_finder(df):
    for index, value in df['exp1_label'].items():
        if pd.isna(value):
            return index
        if value not in [0, 1, 0.0, 1.0]:
            return index
    return len(df)

start_index = index_finder(new_labeled_df)

if start_index >= len(new_labeled_df):
    print("\n The dataset is already fully labeled. No further labeling is required.")
else:
    print(f"\n Continuing labelling from row index: {start_index} out of {len(new_labeled_df)}")

File not found with the given name, creating a new file and starting labeling from the beginning

 Continuing labelling from row index: 0 out of 590


## Create a function that can pass the comment and parent comment through deepseek-r1:8b to label the data based on the given prompt, the temparature is kept as 0 to keep the answers stable even if the same experiment ran multiple times.Using this the model will choose the heighst probability label.

In [5]:
def labeller(parent_comment, comment):
    prompt = f"""
    You are an expert Financial Sentiment Analyst specializing in social media and retail investor forums (e.g., Reddit). 

    Your task is to analyze the relationship between a parent comment and its response comment to detect sarcasm. Financial sarcasm is often subtle, relying on hyperbole, irony, or mocking specific market trends (e.g., "To the moon!" when a stock crashes, or thanking a CEO for losing money).

    Analyze the text provided below. Determine if the response comment is sarcastic based on the context of the parent comment.

    Output rules:
    - Output ONLY a single digit: 1 or 0.
    - Output 1 if the response comment IS sarcastic.
    - Output 0 if the response comment IS NOT sarcastic.
    - Do not include any other text, explanation, punctuation, or spaces.
    - The Parent comment: "{parent_comment}"
    - The Reply comment: "{comment}"

    """
    try:
        result = ollama.chat(
            model='deepseek-r1:8b', 
            messages=[{'role': 'user', 'content': prompt}],
            options={'temperature': 0.0}
        )
        raw_result = result['message']['content'].strip()
        
        if "</think>" in raw_result:
            raw_result = raw_result.split("</think>")[-1].strip()
            
        if "1" in raw_result:
            return 1
        elif "0" in raw_result:
            return 0
        else:
            return 1 if "sarcastic" in raw_result.lower() else 0
            
    except Exception as e:
        raise RuntimeError(f"Ollama Call Error: {e}")



## Once the function is ready, after the checks iterate throgh all the remaining rows inside the dataframe if tsome of the data was already labelled, else start labeling from the first row.

In [6]:

if start_index < len(new_labeled_df):
    print("\nIterating through remaining unlabelled rows via local Ollama instance...")
    start_time = time.time()

    non_labelled_rows = new_labeled_df.iloc[start_index:]

    for index, row in tqdm(non_labelled_rows.iterrows(), total=len(non_labelled_rows), desc="Processing Rows"):
        try:
            prediction = labeller(row['parent_comment'], row['comment'])
            new_labeled_df.at[index, 'exp1_label'] = prediction

            # Print the live update after processing the row
            print(f" Row {index} is successfully processed and labeled.")
            
        except Exception as e:
            print(f"\n Row {index} is skipped, failed with error: {e}. Moving to next row.")
            continue
        
        # After every 10 rows, save the dataframe to the CSV file to ensure progress is not lost
        if index % 10 == 0:
            new_labeled_df.to_csv(exp1_labeled, index=False)

    # Final save
    new_labeled_df.to_csv(exp1_labeled, index=False)

    end_time = time.time()
    total_seconds = end_time - start_time
    hours, minutes, seconds = int(total_seconds // 3600), int((total_seconds % 3600) // 60), int(total_seconds % 60)

    print("\n" + "="*50)
    print(f"Completed labelling the dataset. Total rows processed: {len(non_labelled_rows)}")
    print(f"Results are  saved to: {os.path.abspath(exp1_labeled)}")
    print(f"Total time taken to complete the process is : {hours}h {minutes}m {seconds}s")
    print("="*50)

# # Dataset Verification Summary
# print("\nFinal Model Prediction Distribution:")
# print(new_labeled_df['exp1_label'].value_counts(dropna=False))


Iterating through remaining unlabelled rows via local Ollama instance...


Processing Rows:   0%|          | 1/590 [00:20<3:25:22, 20.92s/it]

 Row 0 is successfully processed and labeled.


Processing Rows:   0%|          | 2/590 [00:33<2:34:32, 15.77s/it]

 Row 1 is successfully processed and labeled.


...

 Row 587 is successfully processed and labeled.


Processing Rows: 100%|█████████▉| 589/590 [2:29:20<00:15, 15.36s/it]

 Row 588 is successfully processed and labeled.


Processing Rows: 100%|██████████| 590/590 [2:29:29<00:00, 15.20s/it]

 Row 589 is successfully processed and labeled.

Completed labelling the dataset. Total rows processed: 590
Results are  saved to: /Users/mariyajose/Documents/Sarcasm/DeepSeek/rq2/exp3_labeled.csv
Total time taken to complete the process is : 2h 29m 29s


In [7]:
print("\n" + "="*50)
print(f"Completed labelling the dataset. Total rows processed: {len(non_labelled_rows)}")
print(f"Results are  saved to: {os.path.abspath(exp1_labeled)}")
print(f"Total time taken to complete the process is : {hours}h {minutes}m {seconds}s")
print("="*50)


Completed labelling the dataset. Total rows processed: 590
Results are  saved to: /Users/mariyajose/Documents/Sarcasm/DeepSeek/rq2/exp3_labeled.csv
Total time taken to complete the process is : 2h 29m 29s
